In [25]:
import openai
import os

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client

import json

In [26]:
qdrant_client = QdrantClient(url = "http://localhost:6333")

/var/folders/r_/3061nb0s7px95mfb__nvwjsc0000gn/T/ipykernel_64977/141586888.py:1: UserWarning: Qdrant client version 1.16.1 is incompatible with server version 1.18.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  qdrant_client = QdrantClient(url = "http://localhost:6333")


### Download all data from Qdrant

In [27]:
all_points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-00",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)

In [28]:
all_points[0][0].payload

{'description': "By John O'Donohue Beauty: The Invisible Embrace Audio CD - March 2004 ",
 'image': 'https://m.media-amazon.com/images/I/31QDhYrrgnL.jpg',
 'rating_number': 496,
 'price': 12.0,
 'average_rating': 4.8,
 'parent_asin': 'B01726HQVK'}

In [29]:
all_context = [
    {"id": data.payload["parent_asin"], "text": data.payload["description"]} for data in all_points[0]
]

In [30]:
all_context

[{'id': 'B01726HQVK',
  'text': "By John O'Donohue Beauty: The Invisible Embrace Audio CD - March 2004 "},
 {'id': 'B07DMPR78Q', 'text': '5 Classic Albums '},
 {'id': 'B08VQYCNHB', 'text': 'Unplugged (CD) '},
 {'id': '5559166928',
  'text': 'Elvis Presley - He Touched Me The Gospel Music - 3pc set '},
 {'id': 'B01BNEZQB4',
  'text': 'GOT7 MAD 4th Mini Album Vertical Version White CD+PhotoCard+Booklet+Tracking Sealed JYP '},
 {'id': 'B0052OM6GA', 'text': 'Be Thankful For What You Got '},
 {'id': 'B001CYKKSU', 'text': 'Chet '},
 {'id': 'B00J3J3KNI',
  'text': 'Enrique Iglesias- Sex and Love Deluxe Extended Edition with 4 Bonus Tracks '},
 {'id': 'B00FY3DD84', 'text': 'Ratt & Roll 8191 by Ratt (1991) Audio CD '},
 {'id': 'B00G2JF3E6',
  'text': 'Best Of Dire Straits & Mark Knopfler: Private Investigations (2CD) by Dire Straits, Mark Knopfler (2005) Audio CD '},
 {'id': 'B003XX6ZIM',
  'text': 'GORDON LIGHTFOOT - endless wire WB 3149 (LP vinyl record) '},
 {'id': 'B003D0L2XS', 'text': 'Lit

#### Render a prompt to generate synthetic Eval reference dataset

In [31]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string"
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string"
                }
            },
            "answer_example": {
                "type": "string"
            },
            "reasoning": {
                "type": "string"
            }
        }
    }
}

SYSTEM_PROMPT = f"""
    I am building a RAG application. I have a collection of 50 chunks of text.
    The RAG application will act as a shopping assistant that can answer questions about the musics we have available.
    I will provide all of the available products to you with IDs of each chunk.
    I want you to come up with 50 questions to which the answers could be grounded in the chunck context.
    The questions shouldimitate a potential real user of this RAG system.
    As an output i need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
    Also provide an example answer to the questioin given the context of the chunks.
    Also, pricide the reason why you chse the chunks to anser the questions.
    Construct 10 questions that could use multiple chunks in the answer.
    Construct 15 qustions that could use single chunk in the answer.
    Construct 5 questions that can't be answered with the available chniks. 

    <OUTPUT JSON SCHEMA>
    {json.dumps(output_schema, indent=2)}
    </OUTPUT JSON SCHEMA>

    I need to be able to parse the json output.
"""


USER_PROMPT = f"""
    Here is the list of chucks, each list element is a dictionary with id and text:
    {all_context}
"""

In [32]:
response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content":USER_PROMPT}
    ],
    reasoning_effort="minimal"
)

print(response.choices[0].message.content)

[
  {
    "question": "What is the title of the music product by John O'Donohue released in March 2004?",
    "chunk_ids": ["B01726HQVK"],
    "answer_example": "Beauty: The Invisible Embrace Audio CD - March 2004",
    "reasoning": "The only chunk mentioning John O'Donohue and a March 2004 release is B01726HQVK, which includes the exact title."
  },
  {
    "question": "Name a multi-disc collection that includes Dire Straits and Mark Knopfler as artists.",
    "chunk_ids": ["B00G2JF3E6"],
    "answer_example": "Best Of Dire Straits & Mark Knopfler: Private Investigations (2CD) by Dire Straits, Mark Knopfler (2005) Audio CD",
    "reasoning": "The chunk B00G2JF3E6 contains the multi-disc Dire Straits collection featuring both artists."
  },
  {
    "question": "Which chunk references Elvis Presley and a gospel music 3-piece set?",
    "chunk_ids": ["5559166928"],
    "answer_example": "Elvis Presley - He Touched Me The Gospel Music - 3pc set",
    "reasoning": "This is the only chunk m

In [33]:
import json

json_output = response.choices[0].message.content
json_output = json.loads(json_output)

In [34]:
points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-00",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B01726HQVK")
            )
        ]
    ),
    limit=100,
    with_payload=True,
    with_vectors=False
)[0]

In [35]:
print(points[0].payload)

{'description': "By John O'Donohue Beauty: The Invisible Embrace Audio CD - March 2004 ", 'image': 'https://m.media-amazon.com/images/I/31QDhYrrgnL.jpg', 'rating_number': 496, 'price': 12.0, 'average_rating': 4.8, 'parent_asin': 'B01726HQVK'}


In [36]:
def get_description(parent_asin: str) -> str:
    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-00",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        ),
        limit=100,
        with_payload=True,
        with_vectors=False
    )[0]

    return points[0].payload["description"]
    


In [37]:
get_description("B01726HQVK")

"By John O'Donohue Beauty: The Invisible Embrace Audio CD - March 2004 "

### Create Eval dataset in Langsmith

In [38]:
client = Client(api_key=os.environ["LANGSMITH_API_KEY"])

In [ ]:
dataset_name = "rag-evaluation-dataset"
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Dataset for evaluating RAG pipeline"
)

In [40]:
for item in json_output:
    # print(item["chunck_ids"])
    client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_description": [get_description(id) for id in item["chunk_ids"]]
        }
    )